# Chapter 22 — Advanced Matrix Operations

Part 8: Advanced Triton operations and production.

**Ladder:** naive triple-loop reference (Method 01) → tiled `tl.dot` GEMM with a K loop (Method 02) → fused bias/activation epilogue and mixed-precision variants (Method 03).

**Thread.**

- **Recurring idea.** A GEMM is one contraction along $K$; every kernel is a scheduling choice around that sum.
- **Chain.** one tile (`tl.dot`) → the global $(M,K)\cdot(K,N)$ rule → a tiled grid → an FP32 accumulator over K panels → tensor cores → mixed precision (FP16/BF16/TF32) → the epilogue → optimization.
- **Invariant.** inner dimensions match, the accumulator stays FP32, and the store is the only cast.

**Study tip.** With $A=\begin{bmatrix}1&2&3\\4&5&6\end{bmatrix}$ and $B=\begin{bmatrix}1&2\\3&4\\5&6\end{bmatrix}$, compute $C_{0,1}=1\cdot2+2\cdot4+3\cdot6=28$ by hand. The full product is $C=\begin{bmatrix}22&28\\49&64\end{bmatrix}$; if it matches, the rest is scheduling.

**Learning objectives**

- State the **contraction rule** $C_{ij}=\sum_k A_{ik}B_{kj}$ and check the inner dimensions before any launch.
- Read a tiled GEMM as a grid of `(BLOCK_M, BLOCK_N)` output tiles, each accumulating over `BLOCK_K` panels.
- Use `tl.dot` with an explicit FP32 accumulator and masks for every non-multiple dimension.
- Distinguish FP16, BF16, and TF32 inputs from the FP32 accumulator, and cast only at the store.
- Fuse a bias and an activation into the GEMM epilogue without materializing an intermediate matrix.
- Estimate FLOPs, bytes, and arithmetic intensity, and explain why tile size changes reuse.

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../ch21_atomics/21_complete.ipynb) · [Next →](../ch23_transformers/23_complete.ipynb)

**Estimated time:** 90–120 min · **Difficulty:** Advanced

**Prerequisites:** Chapter 18 (2-D grids and tiles); Chapter 19 (tiling and vectorization); Chapter 20 (fusion); matrix multiplication and the dot product.

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)


## Chapter map

| § | Topic | Formula | Shape/dtype rule | Kernel/demo |
|---|---|---|---|---|
| **22.1** | `tl.dot()` | $D=\operatorname{dot}(a,b)+c$ | $(B_M,B_K)\cdot(B_K,B_N)\to(B_M,B_N)$ | 2×2 contraction cell |
| **22.2** | Matrix multiplication | $C_{ij}=\sum_k A_{ik}B_{kj}$ | $(M,K)\cdot(K,N)\to(M,N)$ | worked 2×3 example |
| **22.3** | Tiled GEMM | $\text{pid}\mapsto(p_m,p_n)$ | grid $=\lceil M/B_M\rceil\lceil N/B_N\rceil$ | tile-map cell, Method 02 |
| **22.4** | Accumulation | $\mathrm{acc}_{t+1}=\mathrm{acc}_t+A_tB_t$ | accumulator $(B_M,B_N)$ in FP32 | K-panel cell, Method 02 |
| **22.5** | Tensor cores | $D=AB+C$ on small MMA tiles | input tiles multiple of 16, low precision | MMA-count cell |
| **22.6** | Mixed precision | $C=\mathrm{FP32}\left(\sum_k A_{ik}B_{kj}\right)$ | low-precision inputs, FP32 accumulator | Method 03 mixed kernel |
| **22.7** | FP16 and BF16 | 10 vs 7 explicit mantissa bits | same shape; narrower storage and inputs | dtype-rounding cell |
| **22.8** | TF32 | 10 explicit mantissa bits, FP32 exponent | FP32 inputs, `input_precision="tf32"` | tf32 kernel |
| **22.9** | GEMM epilogues | $Y=\operatorname{act}(AB+b^\top)$ | bias $(B_N,)$ broadcasts over rows | bias + ReLU cell, Method 03 |
| **22.10** | GEMM optimization | $\mathrm{FLOPs}=2MNK$ | tile and panel choices change reuse | cost-model and tile-sweep cells |

**One-line thesis.** A GEMM is a contraction along $K$, cut into a grid of
output tiles and a loop of $K$ panels; tensor cores do the small products, FP32
keeps the accumulator honest, and the epilogue is the only place the result is
cast and shaped.


## Concept diagrams

### 1. The contraction: $K$ disappears

```text
        A [M, K]                 B [K, N]
      ┌──────────┐             ┌──────────┐
   i  │  row i   │  ×  k-th    │ column j │  j
      └──────────┘             └──────────┘
                 \             /
                  \           /
                   ▼         ▼
              C[i, j] = Σ_k A[i, k] B[k, j]
```

$M$ and $N$ label the output; $K$ is the only summed axis. Every `tl.dot`
consumes a $(B_M,B_K)$ tile and a $(B_K,B_N)$ tile and produces one
$(B_M,B_N)$ tile.

### 2. The tiled grid

```text
             n = 0        n = 1        n = 2
          ┌───────────┬───────────┬───────────┐
   m = 0  │  pid 0    │  pid 1    │  pid 2    │
          ├───────────┼───────────┼───────────┤
   m = 1  │  pid 3    │  pid 4    │  pid 5    │
          └───────────┴───────────┴───────────┘
             B_N cols    B_N cols    B_N cols
```

With `pid = pm * num_n + pn`, a 1-D launch covers a 2-D grid of output tiles.
Each program owns one tile and writes it once.

### 3. The accumulator

```text
   acc = zeros(B_M, B_N)        FP32
      │
      ├── K panel 0:  acc += dot(A[:, 0:B_K],  B[0:B_K, :])
      ├── K panel 1:  acc += dot(A[:, B_K:2B_K], B[B_K:2B_K, :])
      └── ...
      │
      ▼
   epilogue:  cast / bias / activation
```

The accumulator lives across the whole K loop. It is initialized once, updated
`cdiv(K, B_K)` times, and consumed once.

### 4. The epilogue

```text
   acc [B_M, B_N]  ──►  + bias[None, :]  ──►  act(·)  ──►  cast  ──►  store
                         broadcast over rows              once
```

The epilogue is elementwise on the output tile. It never touches $K$ and never
reads the input tiles again.


In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import pytest
import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, require_gpu, run_if_gpu

print("device:", DEVICE, "| GPU:", HAS_GPU, "| triton:", triton.__version__)


# Part I — Mathematics and Concepts

**Thread.** The ten sections use one card: Connection, Rule, Formula, Example, Where, Triton, Failure, Bridge.

- **Reading order.** 22.1-22.2 fix the contraction; 22.3-22.4 schedule it; 22.5-22.8 select hardware and dtype; 22.9-22.10 consume and optimize the result.
- **Notation.** $M,N$ label the output, $K$ is the summed axis, and $B_M,B_N,B_K$ are the compile-time tile sizes.
- **Check.** Every numerical claim is repeated as an assertion in the code cells.


## 22.1 `tl.dot()`

**Connection.** Builds on the tile ownership of Chapter 19 and the 2-D offsets of Chapter 18. This is the first operation in the book that contracts an axis.

- **Rule.** Multiply a $(B_M,B_K)$ tile by a $(B_K,B_N)$ tile; the result is $(B_M,B_N)$ and the $K$ axis disappears.
- **Formula.** $D=\operatorname{dot}(a,b)+c$ with $D_{ij}=\sum_{k=0}^{K-1}a_{ik}b_{kj}+c_{ij}$; inner sizes must agree exactly, and $B_M,B_N,B_K$ are `tl.constexpr`.
- **Example.** With $A=\begin{bmatrix}1&2\\3&4\end{bmatrix}$ and $B=\begin{bmatrix}5&6\\7&8\end{bmatrix}$, $D=AB=\begin{bmatrix}19&22\\43&50\end{bmatrix}$. Check $D_{1,0}=3\cdot5+4\cdot7=43$.
- **Where.** Every linear layer, attention score, and projection is a `tl.dot`; one call is the smallest complete GEMM, with one tile in and one tile out.
- **Triton.** `tl.dot(a,b)` returns `float32` for FP16/BF16 inputs with an FP32 accumulator; the interpreter accepts small padded tiles, so the arithmetic is checkable without a GPU.
- **Failure.** A mismatched inner dimension is a compile error; a tile too small for the tensor-core path falls back or fails depending on the backend.
- **Bridge.** 22.2 applies this one-tile operation to every output element of a full matrix.


In [ ]:
A22 = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B22 = torch.tensor([[5.0, 6.0], [7.0, 8.0]])
D22 = A22 @ B22

assert A22.shape == B22.shape == D22.shape == (2, 2)
assert D22.tolist() == [[19.0, 22.0], [43.0, 50.0]]
assert D22[1, 0].item() == 3.0 * 5.0 + 4.0 * 7.0
assert D22[1, 0].item() == sum(A22[1, k].item() * B22[k, 0].item()
                              for k in range(2))


## 22.2 Matrix multiplication

**Connection.** Lifts 22.1 from one tile to the full matrix product.

- **Rule.** Sum over the only shared axis: $K$.
- **Formula.** $C_{ij}=\sum_{k=0}^{K-1}A_{ik}B_{kj}$ and $(M,K)\cdot(K,N)\to(M,N)$; also $(AB)^\top=B^\top A^\top$ and $(AB)C=A(BC)$.
- **Example.** With the study-tip matrices, $C=\begin{bmatrix}22&28\\49&64\end{bmatrix}$: $C_{0,0}=22$, $C_{0,1}=28$, $C_{1,0}=49$, $C_{1,1}=64$.
- **Where.** This is the reference definition for Method 01 and every `torch.matmul` check; it also fixes row-major layout: `A(i,k)` at `i*K+k`, `B(k,j)` at `k*N+j`.
- **Triton.** Encode offsets explicitly: `A + rows[:, None]*K + rk[None, :]` and `B + rk[:, None]*N + cols[None, :]`.
- **Failure.** Passing `B` already transposed as `(N,K)` silently changes the contraction; pass `B.T.contiguous()` or explicit strides.
- **Bridge.** 22.3 cuts the output into tiles so each program owns one piece.


In [ ]:
torch.manual_seed(0)
A = torch.randn(4, 5)
B = torch.randn(5, 3)
C = A @ B

assert A.shape == (4, 5) and B.shape == (5, 3) and C.shape == (4, 3)
assert torch.allclose(C.T, B.T @ A.T)

D = torch.randn(3, 2)
assert torch.allclose((A @ B) @ D, A @ (B @ D))

try:
    A @ torch.randn(4, 3)
except RuntimeError:
    pass
else:
    raise AssertionError("K mismatch must raise")


## 22.3 Tiled GEMM

**Connection.** Applies 22.2's global rule one output tile at a time.

- **Rule.** Partition the output into $(B_M,B_N)$ tiles and assign one program per tile.
- **Formula.** $G_M=\lceil M/B_M\rceil$, $G_N=\lceil N/B_N\rceil$, grid $=G_MG_N$, $p_m=\lfloor\text{pid}/G_N\rfloor$, $p_n=\text{pid}\bmod G_N$; rows $=p_mB_M+\{0,\dots,B_M-1\}$ and cols $=p_nB_N+\{0,\dots,B_N-1\}$.

**Example.** For $M=N=4$, $B_M=B_N=2$:

| `pid` | $(p_m,p_n)$ | rows | cols | output |
|---:|---|---|---|---|
| 0 | (0,0) | 0:2 | 0:2 | $C_{00}$ |
| 1 | (0,1) | 0:2 | 2:4 | $C_{01}$ |
| 2 | (1,0) | 2:4 | 0:2 | $C_{10}$ |
| 3 | (1,1) | 2:4 | 2:4 | $C_{11}$ |

- **Where.** One program per output tile gives each tile a private accumulator and one store; this is the layout of most Triton matmul kernels and of the persistent variants.
- **Triton.** `tl.program_id(0)` is the only id needed for a 1-D launch; `tl.cdiv` supplies the ceiling for the grid and the K loop.
- **Failure.** A floor grid misses the last partial tile; a floor division in the pid decode transposes the tile order or assigns a tile twice.
- **Bridge.** 22.4 adds the K loop inside each tile.


In [ ]:
def tile_map(M, N, BM, BN):
    gm = (M + BM - 1) // BM
    gn = (N + BN - 1) // BN
    out = []
    for pid in range(gm * gn):
        pm, pn = divmod(pid, gn)
        out.append((pid, pm, pn,
                    (pm * BM, min((pm + 1) * BM, M)),
                    (pn * BN, min((pn + 1) * BN, N))))
    return out


small = tile_map(4, 4, 2, 2)
assert len(small) == 4
assert small == [
    (0, 0, 0, (0, 2), (0, 2)),
    (1, 0, 1, (0, 2), (2, 4)),
    (2, 1, 0, (2, 4), (0, 2)),
    (3, 1, 1, (2, 4), (2, 4)),
]

tail = tile_map(37, 29, 16, 16)
assert len(tail) == 3 * 2
assert tail[0][3] == (0, 16) and tail[0][4] == (0, 16)
assert tail[-1][3] == (32, 37) and tail[-1][4] == (16, 29)
assert all(r1 - r0 <= 16 for _, _, _, (r0, r1), _ in tail)


## 22.4 Accumulation

**Connection.** Keeps the tile map of 22.3 and adds the loop over K panels.

- **Rule.** Initialize `acc` once, add one panel product per iteration, and consume the accumulator after the loop.
- **Formula.** $\mathrm{acc}_{t+1}=\mathrm{acc}_t+A[:,k_t:k_t+B_K]B[k_t:k_t+B_K,:]$, $\mathrm{acc}_0=0$; after $T=\lceil K/B_K\rceil$ panels, $\mathrm{acc}_T=C$.
- **Example.** Split the study-tip $K=3$ into panels $k\in\{0,1\}$ and $k=2$: $A_0B_0=\begin{bmatrix}7&10\\19&28\end{bmatrix}$ and $A_1B_1=\begin{bmatrix}15&18\\30&36\end{bmatrix}$; their sum is $C=\begin{bmatrix}22&28\\49&64\end{bmatrix}$.
- **Where.** The accumulator is the only state carried across the K loop; it must stay FP32 even when inputs are FP16 or BF16.
- **Triton.** `acc = tl.dot(a,b,acc=acc)` fuses the panel add; `acc += tl.dot(a,b)` is correct but less direct for the compiler.
- **Failure.** An FP16 accumulator loses precision and can overflow; a missing `acc=acc` keeps only the last panel.
- **Bridge.** 22.5 explains the hardware that executes each panel.


In [ ]:
torch.manual_seed(1)
A = torch.randn(5, 7)
B = torch.randn(7, 4)

acc = torch.zeros(5, 4)
panels = []
for k0 in range(0, 7, 3):
    partial = A[:, k0:k0 + 3] @ B[k0:k0 + 3, :]
    panels.append(partial)
    acc = acc + partial

assert len(panels) == 3
assert torch.allclose(acc, A @ B)
assert torch.allclose(panels[0] + panels[1] + panels[2], A @ B)


## 22.5 Tensor cores

**Connection.** Explains the hardware that executes 22.4's panel multiply.

- **Rule.** A tensor core multiplies small MMA tiles, for example $16\times16\times16$, and adds the result to an accumulator.
- **Formula.** $D_{\text{MMA}}=A_{\text{MMA}}B_{\text{MMA}}+C_{\text{MMA}}$; a $B_M\times B_N\times B_K$ dot lowers to $(B_M/16)(B_N/16)(B_K/16)$ MMA operations of $2\cdot16^3$ FLOPs each.
- **Example.** For $B_M=B_N=64$, $B_K=32$: $4\cdot4\cdot2=32$ MMAs and $32\cdot2\cdot16^3=262144=2\cdot64\cdot64\cdot32$ FLOPs.
- **Where.** Tensor cores are why a GEMM can exceed scalar FP32 throughput; they require low-precision inputs and a wider accumulator.
- **Triton.** Write `tl.dot` on tile dimensions that are multiples of the hardware shape; the compiler chooses the instruction and layout.
- **Failure.** An unsupported tile dimension is rejected or silently falls back; very small tiles such as $8\times8$ do not use tensor cores.
- **Bridge.** 22.6 changes the input dtype while keeping the accumulator FP32.


In [ ]:
BM, BN, BK = 64, 64, 32
mma = (BM // 16) * (BN // 16) * (BK // 16)

assert mma == 32
assert mma * (16 ** 3) * 2 == 2 * BM * BN * BK
assert (BM, BN, BK) == (64, 64, 32)


## 22.6 Mixed precision

**Connection.** Adds dtype to the tensor-core path of 22.5.

- **Rule.** Low-precision storage and inputs, FP32 accumulation, one cast at the store.
- **Formula.** $C_{ij}=\operatorname{FP32}(\sum_{k=0}^{K-1}A_{ik}^{\text{low}}B_{kj}^{\text{low}})$ and $C=\operatorname{cast}_{\text{out}}(\mathrm{acc})$.
- **Example.** $1+2^{-10}$ survives FP16 as the next value after $1$; $1+2^{-11}$ is halfway and rounds back to $1$, before any multiply.
- **Where.** FP16/BF16 halve the bytes moved and raise tensor-core throughput; the sum over $K$ stays FP32 because it is the sensitive part.
- **Triton.** `tl.zeros((BM,BN), tl.float32)`, `tl.dot(a,b,acc=acc)`, and `acc.to(C.dtype.element_ty)` at the store.
- **Failure.** FP16 accumulation gives large relative error and can reach `inf`; casting before the epilogue discards precision the epilogue may need.
- **Bridge.** 22.7 compares the two 16-bit formats.


## 22.7 FP16 and BF16

**Connection.** Compares the two 16-bit formats introduced in 22.6.

- **Rule.** Both are 16 bits; FP16 spends 10 bits on the mantissa, BF16 spends 7 and keeps the FP32 exponent.
- **Formula.** ULP at $1$ is $2^{-10}$ for FP16 and $2^{-7}$ for BF16; FP16 max is $65504$, while BF16 shares the FP32 range.

**Format table.**

| Format | Sign | Exponent | Mantissa | ULP at 1 | Range |
|---|---:|---:|---:|---:|---|
| FP16 | 1 | 5 | 10 | $2^{-10}$ | $\approx 6.1\times10^{-5}$ to $65504$ |
| BF16 | 1 | 8 | 7 | $2^{-7}$ | $\approx 1.2\times10^{-38}$ to $3.4\times10^{38}$ |
| FP32 | 1 | 8 | 23 | $2^{-23}$ | $\approx 1.2\times10^{-38}$ to $3.4\times10^{38}$ |

- **Example.** FP16 represents $1+2^{-10}$ but not $1+2^{-11}$; BF16 represents $1+2^{-7}$ but not $1+2^{-8}$.
- **Where.** FP16 suits inference and tight ranges; BF16 suits training because its exponent matches FP32.
- **Triton.** Load the pointer as-is; the tile dtype is the buffer dtype; cast at the store, not before the dot.
- **Failure.** FP16 overflows above $65504$ and underflows near $6.1\times10^{-5}$; BF16 has only about 2.4 decimal digits, so cancellation shows sooner.
- **Bridge.** 22.8 adds TF32, the FP32-input tensor-core format.


In [ ]:
def to_tf32(x):
    bits = x.view(torch.int32) & torch.tensor(-8192, dtype=torch.int32)
    return bits.view(torch.float32)


def test_dtype_precision():
    assert torch.tensor(1.0 + 2.0 ** -10, dtype=torch.float16).item() == 1.0 + 2.0 ** -10
    assert torch.tensor(1.0 + 2.0 ** -11, dtype=torch.float16).item() == 1.0
    assert torch.tensor(1.0 + 2.0 ** -7, dtype=torch.bfloat16).item() == 1.0 + 2.0 ** -7
    assert torch.tensor(1.0 + 2.0 ** -8, dtype=torch.bfloat16).item() == 1.0
    assert to_tf32(torch.tensor(1.0 + 2.0 ** -10)).item() == 1.0 + 2.0 ** -10
    assert to_tf32(torch.tensor(1.0 + 2.0 ** -11)).item() == 1.0
    assert torch.tensor(70000.0, dtype=torch.float16).item() == float("inf")
    print("dtypes: fp16 ulp 2^-10, bf16 ulp 2^-7, tf32 ulp 2^-10")


test_dtype_precision()


## 22.8 TF32

**Connection.** Adds the third tensor-core input format, TF32, to 22.6-22.7.

- **Rule.** TF32 keeps 1 sign, 8 exponent, and 10 explicit mantissa bits; it has FP32's range and FP16's mantissa precision.
- **Formula.** $x_{\text{tf32}}=\operatorname{round}_{10}(x_{\text{fp32}})$.
- **Example.** $1+2^{-10}$ is representable; $1+2^{-11}$ rounds to $1$; $1.5+2^{-11}$ rounds to $1.5$ because the ULP at $1.5$ is $2^{-10}$.
- **Where.** TF32 lets an FP32 GEMM use tensor cores without a source-level conversion; it is the fast default for FP32 `tl.dot` on NVIDIA.
- **Triton.** `input_precision="tf32"` requests the fast path; `input_precision="ieee"` requests full FP32 and is the correctness reference.
- **Failure.** Expecting bitwise agreement with FP32; use tolerance $10^{-2}$ or `"ieee"` for tight tests.
- **Bridge.** 22.9 leaves the K loop and applies the epilogue.


## 22.9 GEMM epilogues

**Connection.** Consumes the accumulator produced by 22.4 after the K loop.

- **Rule.** Apply an elementwise bias and activation to the output tile, then store once.
- **Formula.** $Y_{ij}=\operatorname{act}(C_{ij}+b_j)$ with $C=AB$; the bias $b\in\mathbb{R}^{B_N}$ broadcasts as `bv[None, :]`, so $C+b^\top\in\mathbb{R}^{B_M\times B_N}$.
- **Example.** With $C=\begin{bmatrix}1&-2\\3&4\end{bmatrix}$ and $b=\begin{bmatrix}0.5&-1\end{bmatrix}$, $C+b^\top=\begin{bmatrix}1.5&-3\\3.5&3\end{bmatrix}$ and $\operatorname{ReLU}(C+b^\top)=\begin{bmatrix}1.5&0\\3.5&3\end{bmatrix}$.
- **Where.** Bias plus activation after a linear layer is the standard MLP block; fusion avoids a full $(M,N)$ write and reread.
- **Triton.** Load `bias + cols` with the column mask, add `bv[None, :]`, apply the activation, cast once at the store.
- **Failure.** A bias of shape $(B_M,)$ added as `bv[:, None]` is shape-correct but wrong; indexing by `rows` instead of `cols` is the usual bug.
- **Bridge.** 22.10 asks which tile and panel sizes make this schedule fast.


In [ ]:
C = torch.tensor([[1.0, -2.0], [3.0, 4.0]])
bias = torch.tensor([0.5, -1.0])
base = C + bias
act = torch.relu(base)

assert base.tolist() == [[1.5, -3.0], [3.5, 3.0]]
assert act.tolist() == [[1.5, 0.0], [3.5, 3.0]]
assert torch.broadcast_shapes(C.shape, bias.shape) == (2, 2)

col = torch.tensor([[0.5], [-1.0]])
assert (C + col).tolist() == [[1.5, -1.5], [2.0, 3.0]]


## 22.10 GEMM optimization

**Connection.** Optimizes the schedule built in 22.3-22.9; the arithmetic is already fixed by the contraction.

- **Rule.** Treat FLOPs as fixed and trade tile size, panel depth, warps, and stages for reuse and occupancy.
- **Formula.** $\mathrm{FLOPs}=2MNK$, $Q_{\min}=b(MK+KN+MN)$, $\mathrm{AI}_{\min}=2MNK/Q_{\min}$, and $Q_{\text{tile}}=bMNK(1/B_M+1/B_N)$.
- **Example.** For $M=N=K=1024$ and FP32, $2.147\times10^9$ FLOPs, $Q_{\min}=12.58$ MB, and $\mathrm{AI}_{\min}=170.7$. At $B_M=B_N=64$, $Q_{\text{tile}}=134.2$ MB and $\mathrm{AI}=16.0$; at $128$, $67.1$ MB and $\mathrm{AI}=32.0$.
- **Where.** Every production GEMM compromises between reuse, register pressure, and occupancy; the model gives the direction, the benchmark gives the stop.
- **Triton.** Start square, sweep `BLOCK_K` in $\{16,32,64\}$, then `num_warps` and `num_stages`; change one axis at a time.
- **Failure.** Optimizing before masks and the accumulator are correct; a fast kernel that misses the last tile is not a kernel.
- **Where this leads.** Part II turns the reference into code, Part III builds Method 02, Part IV adds the epilogue and mixed precision, and the performance model predicts which tile to try.


In [ ]:
def test_cost_model():
    M = N = K = 1024
    flops = 2 * M * N * K
    q_min = 4 * (M * K + K * N + M * N)
    ai_min = flops / q_min

    assert flops == 2_147_483_648
    assert q_min == 12_582_912
    assert round(ai_min, 1) == 170.7

    BM = BN = 64
    q_tile = 4 * M * N * K * (1 / BM + 1 / BN)
    assert q_tile == 134_217_728
    assert flops / q_tile == 16.0

    smem_per_stage = 4 * (BM + BN) * 32
    assert smem_per_stage == 16_384
    print("cost model: 2MNK = 2.147 GFLOP; AI 170.7 ideal, 16.0 at 64x64 tiles")


test_cost_model()


## Worked numerical example

### The 2×3 product

**Connection.** Applies 22.2 to a matrix that fits on one screen.

- **Rule.** $K=3$, so each entry is a sum of three products.
- **Formula.** $C_{ij}=A_{i0}B_{0j}+A_{i1}B_{1j}+A_{i2}B_{2j}$.
- **Example.** With $A=\begin{bmatrix}1&2&3\\4&5&6\end{bmatrix}$ and $B=\begin{bmatrix}1&2\\3&4\\5&6\end{bmatrix}$:
  - $C_{0,0}=1\cdot1+2\cdot3+3\cdot5=22$
  - $C_{0,1}=1\cdot2+2\cdot4+3\cdot6=28$
  - $C_{1,0}=4\cdot1+5\cdot3+6\cdot5=49$
  - $C_{1,1}=4\cdot2+5\cdot4+6\cdot6=64$
- **Result.** $C=\begin{bmatrix}22&28\\49&64\end{bmatrix}$, $\mathrm{FLOPs}=24$, FP32 bytes $=64$.
- **Bridge.** The next subsection scales the same arithmetic to four tiles.

### The 4×4 tiled product

**Connection.** Applies 22.3-22.4 with `BLOCK_M=BLOCK_N=BLOCK_K=2`.

- **Rule.** Four output tiles, two K panels per tile.
- **Example.** $A=\begin{bmatrix}1&2&3&4\\5&6&7&8\\9&10&11&12\\13&14&15&16\end{bmatrix}$ and $B=\begin{bmatrix}1&0&1&0\\0&1&0&1\\1&0&0&1\\0&1&1&0\end{bmatrix}$.
- **Tile map.** Each `pid` owns one tile:

| `pid` | $(p_m,p_n)$ | rows | cols | output |
|---:|---|---|---|---|
| 0 | (0,0) | 0:2 | 0:2 | $C_{00}$ |
| 1 | (0,1) | 0:2 | 2:4 | $C_{01}$ |
| 2 | (1,0) | 2:4 | 0:2 | $C_{10}$ |
| 3 | (1,1) | 2:4 | 2:4 | $C_{11}$ |

- **Panel check.** For tile $(0,0)$, both panels multiply by $I$, so $\begin{bmatrix}1&2\\5&6\end{bmatrix}I+\begin{bmatrix}3&4\\7&8\end{bmatrix}I=\begin{bmatrix}4&6\\12&14\end{bmatrix}$.
- **Tiles.** $C_{00}=\begin{bmatrix}4&6\\12&14\end{bmatrix}$, $C_{01}=\begin{bmatrix}5&5\\13&13\end{bmatrix}$, $C_{10}=\begin{bmatrix}20&22\\28&30\end{bmatrix}$, $C_{11}=\begin{bmatrix}21&21\\29&29\end{bmatrix}$.

Assembling them gives

$$
C=\begin{bmatrix}
4&6&5&5\\
12&14&13&13\\
20&22&21&21\\
28&30&29&29
\end{bmatrix}.
$$

- **Bridge.** The next cells verify both products with assertions.


In [ ]:
def test_worked_example():
    A = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
    B = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
    C = A @ B

    assert A.shape == (2, 3) and B.shape == (3, 2) and C.shape == (2, 2)
    assert C.tolist() == [[22.0, 28.0], [49.0, 64.0]]
    assert C[0, 0].item() == 1.0 * 1.0 + 2.0 * 3.0 + 3.0 * 5.0
    assert C[0, 1].item() == 1.0 * 2.0 + 2.0 * 4.0 + 3.0 * 6.0
    assert C[1, 0].item() == 4.0 * 1.0 + 5.0 * 3.0 + 6.0 * 5.0
    assert C[1, 1].item() == 4.0 * 2.0 + 5.0 * 4.0 + 6.0 * 6.0
    assert 2 * 2 * 3 * 2 == 24
    assert 4 * (2 * 3 + 3 * 2 + 2 * 2) == 64


test_worked_example()


In [ ]:
def test_tiled_example():
    A = torch.arange(1.0, 17.0).reshape(4, 4)
    B = torch.tensor([[1.0, 0.0, 1.0, 0.0],
                      [0.0, 1.0, 0.0, 1.0],
                      [1.0, 0.0, 0.0, 1.0],
                      [0.0, 1.0, 1.0, 0.0]])
    C = A @ B
    expected = [[4.0, 6.0, 5.0, 5.0],
                [12.0, 14.0, 13.0, 13.0],
                [20.0, 22.0, 21.0, 21.0],
                [28.0, 30.0, 29.0, 29.0]]
    assert C.tolist() == expected

    tiles = {}
    for pm in range(2):
        for pn in range(2):
            acc = torch.zeros(2, 2)
            for pk in range(2):
                acc += (A[2 * pm:2 * pm + 2, 2 * pk:2 * pk + 2]
                        @ B[2 * pk:2 * pk + 2, 2 * pn:2 * pn + 2])
            tiles[(pm, pn)] = acc
            assert torch.equal(acc, C[2 * pm:2 * pm + 2, 2 * pn:2 * pn + 2])

    assert tiles[(0, 0)].tolist() == [[4.0, 6.0], [12.0, 14.0]]
    assert tiles[(0, 1)].tolist() == [[5.0, 5.0], [13.0, 13.0]]
    assert tiles[(1, 0)].tolist() == [[20.0, 22.0], [28.0, 30.0]]
    assert tiles[(1, 1)].tolist() == [[21.0, 21.0], [29.0, 29.0]]


test_tiled_example()


# Part II — Method 01: the triple-loop reference

**Connection.** Turns 22.2 into the scalar oracle used by every later kernel.

- **Reference loop.** For each `i`, `j`, sum `a[i,k] * b[k,j]` over `k` and write one output element.
- **Purpose.** It is the definition (22.2), one output element at a time, and it is intentionally too slow for a GPU workload.
- **Contraction check.** If `a.shape[1] != b.shape[0]`, the reference raises before doing any work.
- **Bridge.** Method 02 keeps the arithmetic and replaces the loops with one program per tile.


In [ ]:
def naive_matmul(a, b):
    """Method 01: one scalar dot product per output element."""
    M, K = a.shape
    K2, N = b.shape
    if K != K2:
        raise ValueError(f"contraction mismatch: K={K} vs {K2}")

    out = torch.zeros((M, N), dtype=torch.float32, device=a.device)
    for i in range(M):
        for j in range(N):
            total = 0.0
            for k in range(K):
                total += float(a[i, k]) * float(b[k, j])
            out[i, j] = total
    return out


def test_naive_matmul():
    A = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
    B = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
    assert torch.equal(naive_matmul(A, B), A @ B)

    torch.manual_seed(2)
    a = torch.randn(3, 4)
    b = torch.randn(4, 5)
    assert torch.allclose(naive_matmul(a, b), a @ b)

    try:
        naive_matmul(torch.randn(2, 3), torch.randn(4, 5))
    except ValueError:
        pass
    else:
        raise AssertionError("K mismatch must raise")

    print("method 01: triple loop is the correctness oracle")


test_naive_matmul()


# Part III — Method 02: tiled Triton GEMM

## Grid, tile map, and accumulator

**Connection.** Implements 22.3-22.4 with one program per output tile and a K loop.

- **Rule.** Each program owns one output tile, loads one A panel and one B panel per iteration, and stores the tile once.
- **Formula.** $G_M=\lceil M/B_M\rceil$, $G_N=\lceil N/B_N\rceil$, grid $=G_MG_N$, $p_m=\lfloor\text{pid}/G_N\rfloor$, $p_n=\text{pid}\bmod G_N$; $A_{\text{tile}}\in\mathbb{R}^{B_M\times B_K}$, $B_{\text{tile}}\in\mathbb{R}^{B_K\times B_N}$, $\mathrm{acc}\in\mathbb{R}^{B_M\times B_N}$.

**Example.** For the 4×4 product with $B_M=B_N=B_K=2$:

| `pid` | $(p_m,p_n)$ | rows | cols | K panels | `acc` shape |
|---:|---|---|---|---|---|
| 0 | (0,0) | 0:2 | 0:2 | 0:2, 2:4 | 2×2 |
| 1 | (0,1) | 0:2 | 2:4 | 0:2, 2:4 | 2×2 |
| 2 | (1,0) | 2:4 | 0:2 | 0:2, 2:4 | 2×2 |
| 3 | (1,1) | 2:4 | 2:4 | 0:2, 2:4 | 2×2 |

- **Tail.** For $M=37$, $N=29$, $B_M=B_N=16$, the grid is $3\cdot2=6$; the last row tile has 5 live rows and the last column tile has 13 live columns.
- **K loop.** `acc` is initialized once; each panel calls `tl.dot(a,b,acc=acc)`, and `other=0.0` makes a masked K lane contribute nothing.
- **Layout.** `A` is row-major `(M,K)`, `B` is row-major `(K,N)`, `C` is row-major `(M,N)`; the offsets are `rows[:,None]*K + rk[None,:]` and `rk[:,None]*N + cols[None,:]`.
- **Failure.** Missing row, column, or `rk` masks touches out-of-bounds memory.
- **Bridge.** Part IV adds the epilogue and mixed precision to this loop.


In [ ]:
@triton.jit
def gemm_kernel(A, B, C, M, N, K,
                BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    pid = tl.program_id(0)
    num_n = tl.cdiv(N, BN)
    pm = pid // num_n
    pn = pid % num_n

    rows = pm * BM + tl.arange(0, BM)
    cols = pn * BN + tl.arange(0, BN)

    acc = tl.zeros((BM, BN), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BK)):
        rk = k * BK + tl.arange(0, BK)
        a = tl.load(A + rows[:, None] * K + rk[None, :],
                    mask=(rows[:, None] < M) & (rk[None, :] < K),
                    other=0.0)
        b = tl.load(B + rk[:, None] * N + cols[None, :],
                    mask=(rk[:, None] < K) & (cols[None, :] < N),
                    other=0.0)
        acc = tl.dot(a, b, acc=acc, input_precision="ieee")

    tl.store(C + rows[:, None] * N + cols[None, :], acc,
             mask=(rows[:, None] < M) & (cols[None, :] < N))


def gemm(a, b, BM=16, BN=16, BK=16):
    a = torch.as_tensor(a).contiguous()
    b = torch.as_tensor(b).contiguous()
    M, K = a.shape
    K2, N = b.shape
    assert K == K2, f"contraction mismatch: {K} != {K2}"

    c = torch.empty((M, N), device=a.device, dtype=torch.float32)
    grid = (triton.cdiv(M, BM) * triton.cdiv(N, BN),)
    gemm_kernel[grid](a, b, c, M, N, K, BM=BM, BN=BN, BK=BK)
    return c


def test_gemm():
    require_gpu("tiled GEMM")

    A = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], device=DEVICE)
    B = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]], device=DEVICE)
    tiny = gemm(A, B)
    assert tiny.tolist() == [[22.0, 28.0], [49.0, 64.0]]

    torch.manual_seed(3)
    a = torch.randn(37, 20, device=DEVICE)
    b = torch.randn(20, 29, device=DEVICE)
    torch.testing.assert_close(gemm(a, b, BM=16, BN=16, BK=16), a @ b,
                               atol=1e-5, rtol=1e-5)

    print("method 02: tiny tile, K tail, and M/N tail match")


run_if_gpu(test_gemm)


# Part IV — Method 03: epilogue and mixed precision

**Connection.** Extends Method 02's K loop with the work a real layer does after the product.

- **Rule.** Keep the same tile and K loop; add one elementwise pass over the output tile.
- **Formula.** $Y_{ij}=\operatorname{act}(\sum_{k=0}^{K-1}A_{ik}B_{kj}+b_j)$.
- **Where.** The epilogue is elementwise on `acc`, so it adds no reduction and no second pass over the inputs.
- **Failure.** A bias on the wrong axis or a cast before the epilogue is shape-correct but numerically wrong.
- **Bridge.** The first kernel writes the biased product; the second writes bias plus ReLU.


In [ ]:
@triton.jit
def gemm_bias_act_kernel(A, B, bias, C, M, N, K,
                         ACT: tl.constexpr,
                         BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    pid = tl.program_id(0)
    num_n = tl.cdiv(N, BN)
    pm = pid // num_n
    pn = pid % num_n

    rows = pm * BM + tl.arange(0, BM)
    cols = pn * BN + tl.arange(0, BN)

    acc = tl.zeros((BM, BN), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BK)):
        rk = k * BK + tl.arange(0, BK)
        a = tl.load(A + rows[:, None] * K + rk[None, :],
                    mask=(rows[:, None] < M) & (rk[None, :] < K),
                    other=0.0)
        b = tl.load(B + rk[:, None] * N + cols[None, :],
                    mask=(rk[:, None] < K) & (cols[None, :] < N),
                    other=0.0)
        acc = tl.dot(a, b, acc=acc, input_precision="ieee")

    bv = tl.load(bias + cols, mask=cols < N, other=0.0)
    acc = acc + bv[None, :]
    if ACT == 1:
        acc = tl.maximum(acc, 0.0)

    tl.store(C + rows[:, None] * N + cols[None, :], acc,
             mask=(rows[:, None] < M) & (cols[None, :] < N))


def gemm_bias_act(a, b, bias, act=1, BM=16, BN=16, BK=16):
    a = torch.as_tensor(a).contiguous()
    b = torch.as_tensor(b).contiguous()
    bias = torch.as_tensor(bias).contiguous()
    M, K = a.shape
    K2, N = b.shape
    assert K == K2, f"contraction mismatch: {K} != {K2}"

    c = torch.empty((M, N), device=a.device, dtype=torch.float32)
    grid = (triton.cdiv(M, BM) * triton.cdiv(N, BN),)
    gemm_bias_act_kernel[grid](a, b, bias, c, M, N, K, ACT=act,
                               BM=BM, BN=BN, BK=BK)
    return c


def test_epilogue():
    require_gpu("GEMM epilogue")

    torch.manual_seed(4)
    a = torch.randn(35, 18, device=DEVICE)
    b = torch.randn(18, 33, device=DEVICE)
    bias = torch.randn(33, device=DEVICE)
    base = a @ b + bias

    torch.testing.assert_close(gemm_bias_act(a, b, bias, 0), base,
                               atol=1e-5, rtol=1e-5)
    torch.testing.assert_close(gemm_bias_act(a, b, bias, 1), torch.relu(base),
                               atol=1e-5, rtol=1e-5)

    print("method 03: bias and bias+ReLU epilogues match")


run_if_gpu(test_epilogue)


## Mixed precision: FP16 or BF16 in, FP32 accumulation, cast at store

**Connection.** Applies 22.6-22.8 to Method 03.

- **Rule.** The mixed kernel keeps the tiling and changes three things:
  - the input pointers hold FP16 or BF16, so the loaded tiles are narrow;
  - `acc` is still `tl.zeros((BM, BN), tl.float32)`, and `tl.dot` accumulates into it;
  - the store casts once with `acc.to(C.dtype.element_ty)`.
- **Formula.** $\mathrm{acc}\in\mathbb{R}^{B_M\times B_N}_{\text{FP32}}$ and the store maps it to FP16 or BF16.
- **Example.** The reference is `a.float() @ b.float()` because the inputs are already rounded to their storage dtype; the kernel must accumulate their products without adding avoidable error.
- **TF32.** For FP32 inputs, `input_precision="tf32"` is the tensor-core fast path; it rounds each input to 10 mantissa bits, so compare with tolerance $10^{-2}$.
- **Interpreter.** The CPU interpreter exercises the FP16 path and the tiny FP32 cases; BF16 `tl.dot` and TF32 are guarded to a real CUDA device.
- **Failure.** Accumulating in FP16 or casting before the epilogue destroys the precision the FP32 accumulator was meant to protect.
- **Bridge.** The performance section prices this schedule.


In [ ]:
@triton.jit
def mixed_gemm_kernel(A, B, C, M, N, K,
                      BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    pid = tl.program_id(0)
    num_n = tl.cdiv(N, BN)
    pm = pid // num_n
    pn = pid % num_n

    rows = pm * BM + tl.arange(0, BM)
    cols = pn * BN + tl.arange(0, BN)

    acc = tl.zeros((BM, BN), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BK)):
        rk = k * BK + tl.arange(0, BK)
        a = tl.load(A + rows[:, None] * K + rk[None, :],
                    mask=(rows[:, None] < M) & (rk[None, :] < K),
                    other=0.0)
        b = tl.load(B + rk[:, None] * N + cols[None, :],
                    mask=(rk[:, None] < K) & (cols[None, :] < N),
                    other=0.0)
        acc = tl.dot(a, b, acc=acc)

    tl.store(C + rows[:, None] * N + cols[None, :],
             acc.to(C.dtype.element_ty),
             mask=(rows[:, None] < M) & (cols[None, :] < N))


def mixed_gemm(a, b, out_dtype=torch.float32, BM=16, BN=16, BK=16):
    a = torch.as_tensor(a).contiguous()
    b = torch.as_tensor(b).contiguous()
    M, K = a.shape
    K2, N = b.shape
    assert K == K2, f"contraction mismatch: {K} != {K2}"

    c = torch.empty((M, N), device=a.device, dtype=out_dtype)
    grid = (triton.cdiv(M, BM) * triton.cdiv(N, BN),)
    mixed_gemm_kernel[grid](a, b, c, M, N, K, BM=BM, BN=BN, BK=BK)
    return c


@triton.jit
def tf32_gemm_kernel(A, B, C, M, N, K,
                     BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    pid = tl.program_id(0)
    num_n = tl.cdiv(N, BN)
    pm = pid // num_n
    pn = pid % num_n

    rows = pm * BM + tl.arange(0, BM)
    cols = pn * BN + tl.arange(0, BN)

    acc = tl.zeros((BM, BN), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BK)):
        rk = k * BK + tl.arange(0, BK)
        a = tl.load(A + rows[:, None] * K + rk[None, :],
                    mask=(rows[:, None] < M) & (rk[None, :] < K),
                    other=0.0)
        b = tl.load(B + rk[:, None] * N + cols[None, :],
                    mask=(rk[:, None] < K) & (cols[None, :] < N),
                    other=0.0)
        acc = tl.dot(a, b, acc=acc, input_precision="tf32")

    tl.store(C + rows[:, None] * N + cols[None, :], acc,
             mask=(rows[:, None] < M) & (cols[None, :] < N))


def tf32_gemm(a, b, BM=16, BN=16, BK=16):
    a = torch.as_tensor(a).contiguous()
    b = torch.as_tensor(b).contiguous()
    M, K = a.shape
    K2, N = b.shape
    assert K == K2, f"contraction mismatch: {K} != {K2}"

    c = torch.empty((M, N), device=a.device, dtype=torch.float32)
    grid = (triton.cdiv(M, BM) * triton.cdiv(N, BN),)
    tf32_gemm_kernel[grid](a, b, c, M, N, K, BM=BM, BN=BN, BK=BK)
    return c


def test_mixed_precision():
    require_gpu("mixed precision")

    torch.manual_seed(5)
    a = torch.randn(32, 24, device=DEVICE, dtype=torch.float16)
    b = torch.randn(24, 40, device=DEVICE, dtype=torch.float16)
    ref = a.float() @ b.float()

    torch.testing.assert_close(mixed_gemm(a, b), ref, atol=2e-2, rtol=2e-2)
    torch.testing.assert_close(mixed_gemm(a, b, torch.float16).float(), ref,
                               atol=3e-2, rtol=3e-2)

    if torch.cuda.is_available():
        ab, bb = a.bfloat16(), b.bfloat16()
        torch.testing.assert_close(mixed_gemm(ab, bb), ab.float() @ bb.float(),
                                   atol=5e-2, rtol=5e-2)

    print("mixed precision: fp16 -> fp32 accumulator verified")


def test_tf32():
    if not torch.cuda.is_available():
        return

    torch.manual_seed(6)
    a = torch.randn(32, 24, device=DEVICE)
    b = torch.randn(24, 40, device=DEVICE)
    ref = a @ b
    rel = (tf32_gemm(a, b) - ref).abs().max() / ref.abs().max()
    assert rel < 2e-2


run_if_gpu(test_mixed_precision)
run_if_gpu(test_tf32)


# Performance model

**Connection.** Quantifies the trade-offs from 22.10.

- **Rule.** Arithmetic is fixed by the contraction; traffic and reuse depend on the schedule.
- **Formula.** For FP32 with $b=4$ bytes per element:

$$
\mathrm{FLOPs}=2MNK,\qquad
Q_{\min}=b\,(MK+KN+MN),\qquad
\mathrm{AI}_{\min}=\frac{2MNK}{Q_{\min}},\qquad
Q_{\text{tile}}=b\,MNK\left(\frac{1}{B_M}+\frac{1}{B_N}\right).
$$

- **Example.** For $M=N=K=1024$: $2.147\times10^9$ FLOPs, $Q_{\min}=12.58$ MB, and $\mathrm{AI}_{\min}=170.7$ FLOP/byte. At $B_M=B_N=64$, $Q_{\text{tile}}=134.2$ MB and $\mathrm{AI}=16.0$; at $128$, $67.1$ MB and $\mathrm{AI}=32.0$.
- **Where.** The ideal assumes perfect cache reuse; L2 closes part of the gap between the ideal and the tile model.
- **Triton.** One FP32 stage for $B_M=B_N=64$, $B_K=32$ is $4(B_M+B_N)B_K=16$ KiB; larger tiles raise reuse until registers or shared memory run out.
- **Failure.** Reading the tile model as a measurement; it is a direction, not a timing.
- **Bridge.** The final cells check the model and dispatch the tests.


In [ ]:
def test_performance_model():
    M = N = K = 1024
    flops = 2 * M * N * K
    q_min = 4 * (M * K + K * N + M * N)

    assert flops == 2_147_483_648
    assert q_min == 12_582_912
    assert round(flops / q_min, 1) == 170.7
    print("performance: 2MNK FLOPs; ideal AI 170.7 FLOP/byte")


test_performance_model()


In [ ]:
def test_tile_sweep():
    M = N = K = 1024
    flops = 2 * M * N * K
    rows = []
    for tile in (32, 64, 128):
        q = 4 * M * N * K * (1 / tile + 1 / tile)
        rows.append((tile, int(q), flops / q))

    assert rows == [
        (32, 268_435_456, 8.0),
        (64, 134_217_728, 16.0),
        (128, 67_108_864, 32.0),
    ]
    assert rows[0][1] > rows[1][1] > rows[2][1]
    print("tile sweep: 32 -> 64 -> 128 raises AI from 8 to 32 FLOP/byte")


test_tile_sweep()


## Pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| $K$ mismatch | `(M,K)` multiplied by `(N,K)` or `(M,K)` by `(K2,N)` with $K\ne K2$ | assert `a.shape[1] == b.shape[0]` before the launch |
| Tile shape not supported | compile error or slow fallback for tiny/non-multiple MMA tiles | use multiples of 16 on each tile axis and verify once |
| FP16 accumulator | large relative error; `inf` on long sums | `tl.zeros(..., tl.float32)` and `acc=acc` |
| Missing masks | out-of-bounds reads or writes on the last tile | mask rows, cols, and `rk`; use `other=0.0` |
| Transposed `B` | shape-correct but wrong contraction | pass `B` as `(K,N)`, or use explicit strides / `tl.trans` |
| Epilogue broadcast | bias added down columns instead of across columns | bias is `(N,)`; add `bv[None, :]` |
| TF32 precision | mismatch with FP32 reference around the third digit | use `input_precision="ieee"` for reference, or tolerance $10^{-2}$ |
| Cast before the epilogue | bias and activation computed in FP16 | keep `acc` in FP32; cast at the final store |
| Forgetting `acc=acc` | only the last K panel survives | accumulate into the same tile every iteration |

## Self-check

1. For $A\in\mathbb{R}^{3\times5}$ and $B\in\mathbb{R}^{5\times2}$, what is the shape of $AB$, and which axis is contracted?
2. Compute $C_{1,0}$ for the study-tip 2×3 product by hand. Which three products contribute?
3. With $M=N=4$ and $B_M=B_N=2$, list the four `(pid, pm, pn)` triples and the rows/cols each program owns.
4. Why does the K loop initialize `acc` once, outside the loop? What happens if `acc` is reinitialized inside?
5. A tile has $B_M=B_N=64$, $B_K=32$. How many $16^3$ MMA operations does it contain, and how many FLOPs is that?
6. Why is the accumulator FP32 when the inputs are FP16? Give both a precision reason and a range reason.
7. What is the arithmetic intensity of a $1024^3$ FP32 GEMM under the ideal traffic model? What does a $64\times64$ tile model give?
8. In the epilogue, why is the bias indexed by `cols` and added as `bv[None, :]`?

## Summary

- A GEMM contracts the $K$ axis: $(M,K)\cdot(K,N)\to(M,N)$, and the inner sizes must match exactly.
- Tiling assigns one `(B_M,B_N)` output tile to each program and loops over `cdiv(K,B_K)` K panels.
- The accumulator is FP32 and lives across the whole K loop; `tl.dot(a,b,acc=acc)` fuses the panel add.
- Tensor cores operate on small MMA tiles and require low-precision inputs with a wider accumulator.
- The epilogue is elementwise on the output tile: bias broadcasts across columns, activation is applied once, and the cast happens at the store.
- FLOPs are $2MNK$; traffic and arithmetic intensity are controlled by tile size, K-panel depth, and cache reuse.

**Where this leads.** Chapter 23 builds attention and transformer blocks from these GEMMs, epilogues, and reductions; Chapter 24 adds autotuning, benchmarking, and roofline analysis.

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../ch21_atomics/21_complete.ipynb)


# Exercises

1. **Contraction by hand.** For $A=\begin{bmatrix}2&1&0\\1&3&2\end{bmatrix}$ and
   $B=\begin{bmatrix}1&4\\2&0\\1&1\end{bmatrix}$, compute all four entries of
   $AB$ and the FLOP count.
2. **Tile map.** For $M=70$, $N=45$, $B_M=32$, $B_N=16$, list the grid size,
   the last row tile, and the last column tile. Which tiles are partial?
3. **K panels.** Rewrite the worked 2×3 example with $B_K=1$ and show the three
   accumulator updates. Does the final accumulator depend on the panel order in
   exact arithmetic?
4. **Mask audit.** Remove the `rk < K` mask from the Method 02 kernel and explain
   which read goes out of bounds for $K=20$, $B_K=16$.
5. **Accumulator dtype.** Change `acc` to `tl.float16` in the mixed kernel and
   measure the error on a $128\times128\times128$ FP16 product. At what $K$ does
   the result become unusable?
6. **Bias axis.** Write the wrong epilogue that adds `bv[:, None]` with a bias of
   length $M$. Construct a small case where the shape is still correct and the
   values are wrong.
7. **TF32 tolerance.** Compare `input_precision="ieee"` and `"tf32"` on a random
   $64\times64\times64$ FP32 product. Report the maximum absolute and relative
   errors.
8. **Traffic.** For $M=N=K=4096$ and FP32, compute $Q_{\min}$ and the tile
   traffic at $B_M=B_N=64$ and $128$. Which tile is closer to the ideal?
9. **Epilogue fusion.** Add a GELU activation to the epilogue using the tanh
   approximation, and compare it with `torch.nn.functional.gelu`. Which
   tolerance is appropriate?
10. **Autotune.** Sweep `BLOCK_M`, `BLOCK_N`, `BLOCK_K`, `num_warps`, and
    `num_stages` on a real GPU. Keep the correctness assertion in the loop and
    report the best configuration for a $1024^3$ FP16 product.

Hints are in [`../../docs/SOLUTIONS.md`](../../docs/SOLUTIONS.md).


In [ ]:
print("Running CPU tests ...")
for fn in (test_worked_example, test_tiled_example, test_dtype_precision,
           test_naive_matmul, test_cost_model, test_performance_model,
           test_tile_sweep):
    fn()

print("\nRunning GPU tests (skipped without CUDA/ROCm) ...")
for fn in (test_gemm, test_epilogue, test_mixed_precision, test_tf32):
    run_if_gpu(fn)
